In [ ]:
!nvidia-smi

Mon Apr  6 23:41:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "pysam", "pybigwig", "levenshtein", "transformers==4.52", "pymoo"])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'pysam', 'pybigwig', 'levenshtein', 'transformers==4.52', 'pymoo'], returncode=0)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive') # for colab
sys.path.insert(0, '/content/drive/MyDrive/nucleoformer') # for modules
os.chdir('/content/drive/MyDrive/nucleoformer')

Mounted at /content/drive


In [ ]:
from aligner import GeneLoader
from HMM import HMM
from genetic_algorithm import NSGA_II

from sentence_transformers import SentenceTransformer
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoConfig, AutoModelForMaskedLM
import transformers

import numpy as np
from tqdm import tqdm
import polars as pl
import re
import gc

from concurrent.futures import ThreadPoolExecutor
import threading
import time
import copy
import pynvml
from pymoo.indicators.hv import HV

In [ ]:
LOADER = GeneLoader("hg38_primary", "genome")
HMM_MODEL = HMM(n_states=len(LOADER.states), k=6, n_bases=6, model_name="supervised_full")

In [ ]:
def load_model():
  model_name = "InstaDeepAI/nucleotide-transformer-2.5b-multi-species"
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForMaskedLM.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda").eval()
  return model, tokenizer

model, tokenizer = load_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/278M [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.91G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
objectives = {
    "cos_dist": {"reverse": True},
    "ham_dist": {"reverse": False},
    # "hmm_prob": {"reverse": False}
}

CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.02
ITERATIONS = range(500)

name = "cos_ham_rev_jacobian"

In [ ]:
def get_idx(entry):
    filename = entry.split(".")[0]
    number = filename.split("_")[-1]
    return int(number)

df = LOADER.nt_map.with_row_index("idx").filter(pl.col("len") <= 1e4)
df_binned = df.with_columns(pl.col("len").cut(breaks=[500, 2000, 4000, 6000, 10000]).alias("len_bin"))
df_binned = df_binned.filter(pl.col("len") >= 500)
df_stratified = df_binned.group_by(["state", "len_bin"])
to_keep = df_binned.group_by(["state", "len_bin"]).len().filter(pl.col("len") >= 50)
binned_data = df_binned.join(to_keep.select(["state", "len_bin"]), on=["state", "len_bin"])
bins = sorted(binned_data["len_bin"].unique().to_list(),
              key=lambda x: float(x.split(",")[0].strip("(-inf").strip("(") or "-inf"))

processed = set([get_idx(entry) for entry in os.listdir("popns")])

state_counts = binned_data.group_by("state").len()
state_weights = state_counts.with_columns((1 / pl.col("len")).alias("weight"))

filtered = binned_data.filter(~pl.col("idx").is_in(processed)).filter(pl.col("len_bin") == bins[1])
filtered = filtered.join(state_weights.select(["state", "weight"]), on="state")
indices = np.array(filtered["idx"])

weights = filtered["weight"].to_numpy()
weights = weights / weights.sum()

sample_indices = np.random.choice(filtered["idx"].to_numpy(), size=5000, replace=False, p=weights)

In [ ]:
# def monitor_gpu():
#     pynvml.nvmlInit()
#     handle = pynvml.nvmlDeviceGetHandleByIndex(0)
#     while not stop_monitor.is_set():
#         util = pynvml.nvmlDeviceGetUtilizationRates(handle)
#         mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
#         sys.stdout.write(f"{util.gpu}%, {mem.used // 1024**2} MiB\n")
#         sys.stdout.flush()
#         time.sleep(0.5)

In [ ]:
def compute_hypervolume(popn, objectives, ref_point=None):
    front = np.array([[e["fitness"][obj] for obj in objectives]
                      for e in popn if e["live"]])

    if len(front) < 2:
        return 0

    signs = np.array([-1 if not config["reverse"] else 1 for config in objectives.values()])
    front = front * signs

    if ref_point is None:
        ref_point = front.max(axis=0) + 0.1

    hv = HV(ref_point=ref_point)
    return hv(front)

In [ ]:
entry = LOADER.get_idx(sample_indices[0])
input_ids = tokenizer(entry["seq"])["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(input_ids)
special = [t for t in tokens if t in tokenizer.all_special_tokens]
non_special = [t for t in tokens if t not in tokenizer.all_special_tokens]
print(f"total tokens: {len(tokens)}")
print(f"special tokens: {len(special)}")
print(f"non-special tokens: {len(non_special)}")
print(f"sequence length: {len(entry["seq"])}")
print(tokens[:10])
print(tokens[-10:])

total tokens: 201
special tokens: 1
non-special tokens: 200
sequence length: 1200
['<cls>', 'TCGCCC', 'ACCCAG', 'TCTAAG', 'TGTTTT', 'AATTTT', 'TATCTT', 'TTACAA', 'CAGCTT', 'CTAATT']
['GCTTCT', 'CTTGCT', 'GGGGTG', 'TCGATT', 'CGGGAG', 'GGCTGA', 'GGGCGC', 'GGCCGA', 'GAGAAC', 'GGGGCG']


In [ ]:
print(sample_indices)
for index in sample_indices:
    t_total = time.time()

    GA = NSGA_II(f"{name}_{index}", objectives, hmm=HMM_MODEL,
                 loader=LOADER, pop_cap=100, idx=index,
                 model=model, tokenizer=tokenizer, use_jacobian=True,
                 batch_size=12)
    print(f"seq length: {len(GA.master['seq'])}")

    GA.load_popn()
    for i in range(len(GA.popn)):
        GA.random_mutate_entry(i)

    prev_hv = 0
    null = 0
    t_cross, t_update, t_trunc = 0, 0, 0
    for i in tqdm(ITERATIONS):

        t0 = time.time()
        GA.cross_mutate_random(CROSSOVER_RATE, MUTATION_RATE)
        t_cross += time.time() - t0

        t0 = time.time()
        GA.update_entries()
        t_update += time.time() - t0

        t0 = time.time()
        GA.truncate_popn()
        t_trunc += time.time() - t0

        if i % 10 == 0:
            gc.collect()
            t0 = time.time()
            hv = compute_hypervolume(GA.popn, objectives)
            t_hv = time.time() - t0
            sys.stdout.write(f"hv={hv:.4f} hv_time={t_hv:.4f}s\n"); sys.stdout.flush()
            if hv > prev_hv + 1e-6:
                prev_hv = hv
                null = 0
            else:
                null += 1
            if null >= 10:
                break


        n_iter = i + 1
        # sys.stdout.write(f"avg cross_mutate: {t_cross/n_iter:.3f}s\n"); sys.stdout.flush()
        # sys.stdout.write(f"avg update_entries: {t_update/n_iter:.3f}s\n"); sys.stdout.flush()
        # sys.stdout.write(f"avg truncate_popn: {t_trunc/n_iter:.3f}s\n"); sys.stdout.flush()
        # sys.stdout.write(f"total time: {time.time()-t_total:.1f}s\n"); sys.stdout.flush()

    GA.save_popn(write=True)

# stop_monitor.set()
# monitor_thread.join()

[385066 577878 592697 ... 121248 438300 657224]
seq length: 1200


  0%|          | 0/500 [00:00<?, ?it/s]

hv=0.0123 hv_time=0.0004s


  2%|▏         | 10/500 [00:23<16:10,  1.98s/it]

hv=0.0244 hv_time=0.0004s


...

 28%|██▊       | 140/500 [03:06<07:43,  1.29s/it]

hv=0.1635 hv_time=0.0004s


 30%|███       | 150/500 [03:19<07:29,  1.29s/it]

hv=0.1785 hv_time=0.0004s


 31%|███       | 156/500 [03:27<07:26,  1.30s/it]

In [ ]:
# stop_monitor = threading.Event()
# monitor_thread = threading.Thread(target=monitor_gpu)
# monitor_thread.start()

start_time = time.time()
completed = 0
lock = threading.Lock()

def run_ga(args):
    chunk, delay = args
    time.sleep(delay)
    for index in chunk:
        global completed

        # model, tokenizer = load_model()
        sys.stdout.write(f"processing index: {index}\n"); sys.stdout.flush()

        GA = NSGA_II(f"{name}_{index}", objectives,
                     hmm=HMM(n_states=len(LOADER.states), k=6, n_bases=6, model_name="supervised_full"),
                     loader=LOADER,
                     pop_cap=100, idx=index,
                     model=model,
                     tokenizer=tokenizer)
        GA.load_popn()
        for i in range(len(GA.popn)):
            GA.random_mutate_entry(i)

        t_total = time.time()
        t_cross, t_update, t_trunc = 0, 0, 0
        n_iter = len(ITERATIONS)

        for i in ITERATIONS:
            t0 = time.time(); GA.cross_mutate_random(CROSSOVER_RATE, MUTATION_RATE); t_cross += time.time() - t0
            t0 = time.time(); GA.update_entries(); t_update += time.time() - t0
            t0 = time.time(); GA.truncate_popn(); t_trunc += time.time() - t0

        GA.save_popn(write=True)

        with lock:
            completed += 1
            elapsed = time.time() - start_time
            rate = completed / elapsed
            remaining = (len(sample_indices) - completed) / rate if rate > 0 else 0
            sys.stdout.write(f"[{completed}/{len(sample_indices)}] seq_len={len(GA.master['seq'])} | cross={t_cross/n_iter:.3f}s | update={t_update/n_iter:.3f}s | trunc={t_trunc/n_iter:.3f}s | ga_time={time.time()-t_total:.1f}s | eta={remaining:.1f}s\n")
            sys.stdout.flush()

n_threads = 8
chunks = np.array_split(sample_indices, n_threads)
args = [(chunk, i * 0.5) for i, chunk in enumerate(chunks)]
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    executor.map(run_ga, args)

In [ ]:
stop_monitor.set()
monitor_thread.join()